<!-- your logo here! -->
<img src="https://ac.auriasolutions.com/library/images/auria_logo_sm.jpg"/>
<img src="https://github.com/bionadmin/distrib/raw/main/core.png"/>

# Attempt to update the Gold schema without losing data.

#### Debug options

In [1]:
debug_mode = False # Include debug select and print statements
force_remount = False #Normally mount points are left alone if they exists. Setting this to true will replace the mountpoints.
force_rebuild_of_all_tables = False

StatementMeta(, 786bdf9f-4c26-46f7-8ea4-9957519b7df9, 3, Finished, Available, Finished, False)

#### Some items we need...

In [2]:
from datetime import datetime
now = datetime.now()
tag = now.strftime("%Y%m%d%H%M%S")
print(tag)

useManagedTables = True
unmanagedArchiveCopies = 1
managedTablesKeepOldTableVersions = False
rebuildTableWhenColumnOrderChanges = True

destination_database_name="LH_BI_GOLD"
spark.conf.set("sess.NB_GOLD_Default_Table_destination_database_name", destination_database_name)



dest_base_path= "<Dest Folder>"

dest_additional_path="Gold/"
dest_base_path=dest_base_path.replace("<Dest Folder>",dest_additional_path)
destination_file_folder="datamarts/sales"#set to None if no subfolder is desired. Include only internal slashes in the path.
if destination_file_folder==None or destination_file_folder=="":
    destination_database_default_path= dest_base_path
else:
    destination_database_default_path= dest_base_path + "/"+destination_file_folder
destination_filepath=destination_database_default_path+"/"

spark.conf.set("sess.NB_GOLD_Default_Table_destination_database_default_path", destination_database_default_path)

StatementMeta(, 786bdf9f-4c26-46f7-8ea4-9957519b7df9, 4, Finished, Available, Finished, False)

20260226152418


#### Misc. Spark Options
If generating code, edit these in the template settings for consistency<br>
across all notebooks.

In [3]:
spark.conf.set("spark.sql.parquet.vorder.enabled","false")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled","true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize","1073741824")

StatementMeta(, 786bdf9f-4c26-46f7-8ea4-9957519b7df9, 5, Finished, Available, Finished, False)

In [4]:
%%sql
SET ANSI_MODE = TRUE

StatementMeta(, 786bdf9f-4c26-46f7-8ea4-9957519b7df9, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [5]:
def TableArraysAreEqual(old, new):
    if (len(old)!=len(new)):
        return False
    outerIndex=0
    for rowNew in new:
        innerIndex=0
        found = False
        for rowOld in old:
            if rowNew["col_name"]==rowOld["col_name"] and rowNew["data_type"]==rowOld["data_type"] and (rowNew["comment"] or "")==(rowOld["comment"] or ""):
                found=True
                break
            innerIndex=innerIndex+1
        if not found:
            if debug_mode:
                    print("Table structure has changed. Rebuilding table.")
            return False
        if rebuildTableWhenColumnOrderChanges and innerIndex!=outerIndex:
            if debug_mode:
                    print("Column Order has changed. Rebuilding table.")
            return False  
        outerIndex=outerIndex+1
    return True

def GetColumnByComment(arr,comment):
    if comment == "" or comment is None:
        return None
    for row in arr:
        if (row["comment"] is not None):
            if (comment in row["comment"]):
                return row
    return None

def GetColumnByName(arr, name):
    for row in arr:
        if (name == row["col_name"]):
            return row
    return None

def GetColumnByCommentThenName(arr, comment, name):
    row = GetColumnByComment(arr,comment)
    if row is None:
        row = GetColumnByName(arr,name)
    return row

def GenerateTableCreate(new, databaseName, tableName, partitions,location):

    stmt = "CREATE OR REPLACE TABLE "+databaseName+"."+tableName
    stmt=stmt+"\r\n("
    for row in new:
        stmt = stmt+"\r\n\t`"+row["col_name"]+"` "+row["data_type"]
        if row["comment"]:
            stmt = stmt +" COMMENT \""+row["comment"]+"\","
        else:
            stmt = stmt +","
    stmt=stmt[0:len(stmt)-1]#get rid of last trailing comma
    stmt=stmt+"\r\n)"  
    stmt=stmt+"\r\nUSING DELTA"
    if partitions is not None:
        if len(partitions)>0:
            stmt=stmt+"\r\nPARTITIONED BY("
            firstPartition=True
            for par in partitions:
                if not firstPartition:
                    stmt+=","
                firstPartition=False
                stmt+="`"+par+"`"
            stmt=stmt+")"
    if not useManagedTables:
        stmt=stmt+"\r\nLOCATION \""+location+"\""
    if debug_mode:
        print (stmt)
    return stmt

def GenerateTableCopy(old, new, databaseName, oldTableName, newTableName):
    stmt = "INSERT INTO "+databaseName+"."+newTableName
    stmt=stmt+"\r\n("
    for row in new:
        stmt = stmt+"\r\n\t`"+row["col_name"]+"`,"
    stmt=stmt[0:len(stmt)-1]#get rid of last trailing comma
    stmt=stmt+"\r\n)"   
    stmt=stmt+"SELECT "
    for row in new:
        oldRow = GetColumnByCommentThenName(old,row["comment"],row["col_name"])
        if oldRow is None:
            stmt=stmt+"\r\n\tNULL AS `"+row["col_name"]+"`,"
        else:
            stmt=stmt+"\r\n\tCAST(`"+oldRow["col_name"]+"` AS "+row["data_type"]+") AS `"+row["col_name"]+"`,"
    stmt=stmt[0:len(stmt)-1]#get rid of last trailing comma
    stmt=stmt+"\r\nFROM "+databaseName+"."+oldTableName
    if debug_mode:
        print (stmt)
    return stmt
 
def GenerateTableRename(databaseName, oldTableName, newTableName):
    stmt = "ALTER TABLE "+databaseName+"."+oldTableName +" RENAME TO "+newTableName
    if debug_mode:
        print (stmt)
    return stmt



def RemoveNonColumns(arr):
    returnArr = []
    for row in arr:
        if row["col_name"] == "# Partitioning":#Old way
            break  # we've gone past the columns. all done
        if row["col_name"] == "# Partition Information":#New Way
            break  # we've gone past the columns. all done
        if row["col_name"] == "# col_name":#New Way
            break  # we've gone past the columns. all done
        if row["data_type"] != "" and row["data_type"] is not None:
            returnArr.append(row)
    return returnArr
  
def TableExists(databaseName, tableName):
    table_list=spark.sql("show tables in "+databaseName)
    table_name=table_list.filter(table_list.tableName==tableName.lower()).collect()
    if table_name:
        return True
    else:
        return False
    
def UnmanagedRenameFolder(oldlocation,newlocation):
    if useManagedTables:
        return
    notebookutils.fs.mv(oldlocation, newlocation, True)

def UnmanagedDropTable(databaseName, tableName):
    spark.sql("DROP TABLE "+databaseName+"."+tableName)

def ManagedDropTable(databaseName, tableName):
    spark.sql("DROP TABLE "+databaseName+"."+tableName)
    
def UpdateTableStructure(old, new, databaseName, tableName, partitions, location, force):
    
    if (not force) and TableArraysAreEqual(old, new):
        if debug_mode:
            print ("Tables are the same. No need to update.")
        return
    stmt = GenerateTableCreate(new,databaseName, tableName+"_"+tag+"_TMP", partitions,location+"_"+tag+"_TMP")
    spark.sql(stmt)
    stmt = GenerateTableCopy(old,new,databaseName, tableName,tableName+"_"+tag+"_TMP")
    spark.sql(stmt)
    if useManagedTables:
        #With managed tables, the folder gets renamed with the table so this is all we need. Nice and simple.
        stmt = GenerateTableRename(databaseName,tableName, tableName+"_"+tag)
        spark.sql(stmt)
        stmt = GenerateTableRename(databaseName, tableName+"_"+tag+"_TMP",tableName)
        spark.sql(stmt)
        if not managedTablesKeepOldTableVersions:
            ManagedDropTable(databaseName, tableName+"_"+tag)
    if not useManagedTables:
        #with unmanaged, we need to rename the folder too so we don't have the new table's data sitting in a folder with a weird name...
        #basically since these are unmanaged we move the data around and then recreate the table which just updates the meta-data
        UnmanagedDropTable(databaseName,tableName)#drop our two tables.
        UnmanagedDropTable(databaseName,tableName+"_"+tag+"_TMP")#drop our two tables.
        UnmanagedRenameFolder(location,location+"_"+tag+"_PreviousVersion")#move the old version of the table's data out to a previous version folder.
        UnmanagedRenameFolder(location+"_"+tag+"_TMP",location)#move the new tables data into the folder with the basic table name
        stmt=GenerateTableCreate(new,databaseName, tableName, partitions,location)#create the table against the new path.
        if debug_mode:
            print(stmt)
        spark.sql(stmt)
        
def GetPartitions(collectedDescription):
    returnArr = []
    columns = RemoveNonColumns(collectedDescription)
    foundPartitionMarker = False
    newWay = False
    for row in collectedDescription:
        if row["col_name"] == "# Partitioning":  #older way
            foundPartitionMarker = True
            continue
        if row["col_name"] == "# Partition Information":  # newer way
            foundPartitionMarker = True
            newWay=True
            continue
        else:
            if not foundPartitionMarker:
                continue
        # if we made it this far, we are on the partitions section of the description.
        if row["col_name"] == "Not partitioned":
            return returnArr
        if newWay:
            potentialName = row["col_name"]
            if not potentialName.startswith("#"):
                for col in columns:
                    # confirm we are looking at a partition column
                    if col["col_name"].lower() == potentialName.lower():
                        returnArr.append(potentialName)
                        break
        else:
            if row["col_name"].startswith("Part "):  # partitions show up this way.
                potentialName = row[
                    "data_type"
                ]  # partition column names are in the type column.
                for col in columns:
                    # confirm we are looking at a partition column
                    if col["col_name"].lower() == potentialName.lower():
                        returnArr.append(potentialName)
                        break
    return returnArr


def StringArraysHaveSameElementsCI(one,two):
    if len(one)!=len(two):
        return False
    for strone in one:
        found = False
        for strtwo in two:
            if strone.lower()==strtwo.lower():
                found=True
                break
        if not found:
            return False
    return True
    
def RemoveOldArchiveCopies(tableName, filePath):
    if not useManagedTables:
        if unmanagedArchiveCopies>-1:#-1 means keep all copies!
            directoriesToParse=[]
            for fileinfo in notebookutils.fs.ls(filePath):
                name = fileinfo.name
                if name.startswith(tableName) and len(name)==len(tableName)+32 and name.endswith("_PreviousVersion/"):
                    if debug_mode:
                        print (fileinfo.path)
                    directoriesToParse.append(fileinfo.path)
            directoriesToParse.sort(reverse=True)
            filesToKeep = unmanagedArchiveCopies
            for directory in directoriesToParse:
                if filesToKeep>0:
                    filesToKeep=filesToKeep-1
                    continue
                else:
                    if debug_mode:
                        print("Removing archive directory "+directory)
                    notebookutils.fs.rm(directory,True)
                    
def TableIsManaged(databaseName, tableName):
    df = spark.sql("DESCRIBE EXTENDED "+databaseName+"."+tableName)
    arr = df.collect()
    foundDetailed = False
    for row in arr:
        if row["col_name"]=="# Detailed Table Information":
            foundDetailed=True
            continue
        if not foundDetailed:
            continue
        if row["col_name"]=="Type":
            if row["data_type"]=="MANAGED":
                return True
    return False

def ProcessTable(databaseName, tableName, comparisonArray, partitions, path):
    existingTableIsManaged=False
    tableExists = TableExists(databaseName,tableName)
    if tableExists:
        if debug_mode:
            print ("Table Exists")
        description = spark.sql("DESCRIBE "+databaseName+"."+tableName)
        oldpartitions = GetPartitions(description.collect())
        force = not StringArraysHaveSameElementsCI(oldpartitions,partitions)#rewrite table if partitions changed
        if force_rebuild_of_all_tables:
            force = True
        existingTableIsManaged = TableIsManaged(databaseName,tableName)
        if not force:  
            if (useManagedTables and not existingTableIsManaged) or (not useManagedTables and existingTableIsManaged):
                force = True
                if debug_mode:
                    if useManagedTables:
                        print("Converting tables from unmanaged to managed...")
                    else:
                        print("Converting tables from managed to unmanaged...")
        cols = RemoveNonColumns(description.collect())
        UpdateTableStructure(cols, comparisonArray, databaseName,tableName,partitions,path,force)
    else:
        if debug_mode:
            print ("Table Does not Exist. Creating Table...")
        stmt = GenerateTableCreate(comparisonArray,databaseName,tableName,partitions,path)
        spark.sql(stmt)

    if not existingTableIsManaged:
        RemoveOldArchiveCopies(tableName,destination_filepath)

StatementMeta(, 786bdf9f-4c26-46f7-8ea4-9957519b7df9, 7, Finished, Available, Finished, False)

## Master for creating table

In [ ]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
    ('PCN',                       'int',            ''), 
    ('Price_Key',                 'int',            ''), 
    ('PO_Line_Key',               'int',            ''), 
    ('Effective_Date',            'timestamp',      ''), 
    ('Part_Status',               'string',         ''), 
    ('Price',                     'decimal(16,6)',  ''), 
    ('Active',                    'smallint',       ''), 
    ('Expiration_Date',           'timestamp',      ''), 
    ('Unit',                      'string',         ''), 
    ('Add_Date',                  'timestamp',      ''), 
    ('Update_Date',               'timestamp',      ''), 
    ('Cost',                      'decimal(16,6)',  ''), 
    ('Margin',                    'decimal(16,6)',  ''), 
    ('Part_Cost',                 'decimal(16,6)',  ''), 
    ('Price_In_Effect',           'boolean',        ''), 
    ('Price_In_Effect_Basis',     'decimal(16,6)',  ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),

    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "test_plex"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_GOLD",
    table_alias,
    comparisonArray,
    partitions,
    path
)


## Process factshipment

In [6]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('shipmentKey',         'binary',  ''),
('shipmentKeyInt',         'bigint',  ''),
('PCN','int',''),
('Customer_No','String',''),
('Source','String',''),
('Customer_Code','String',''),
('CustomerAddressKey','binary',''),
('CustomerAddressKeyInt',         'bigint',  ''),
('Part_Key','int',''),
('part_masterKey','binary',''),
('part_masterKeyInt', 'bigint',  ''),
('Customer_Part_Key','binary',''),
('name','string',''),
('Ship_Date','timestamp',''),
('Part_No','string',''),
('Customer_Part_No','String',''),
('Quantity_Shipped','decimal(16,4)',''),
('price','decimal(16,4)',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),

    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "factshipment"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_GOLD",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 786bdf9f-4c26-46f7-8ea4-9957519b7df9, 8, Finished, Available, Finished, False)

## Prcoess facton_factrevenue

In [7]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('System','string',''),
('Version','string',''),
('Facton_Project_ID','int',''),
('PlantKey','binary',''),
('PlantKeyInt',         'bigint',  ''),
('bom_approved_workcenterKey',         'binary',  ''),
('bom_approved_workcenterKeyInt',         'bigint',  ''),
('exploded_bomKey',         'binary',  ''),
('exploded_bomKeyInt',         'bigint',  ''),
('part_masterKey',         'binary',  ''),
('part_masterKeyInt',         'bigint',  ''),
#('Location','string',''),
('Part_group_key','string',''),
('Product_Group','string',''),
('Product_type_key','string',''),
#('Customer','string',''),
('zero_level_desc','string',''),
('Zero_level_part_desc','string',''),
('level','string',''),
('parent_part_no','string',''),
('parent_part_no_desc','string',''),
('part_no','string',''),
('Description','string',''),
('Revision','string',''),
('Cost_Type','string',''),
('Cost_Sub_Type','string',''),
('Cost','decimal(16,6)',''),
('Quantity','decimal(16,6)',''),
('Operation_No','int',''),
('Operation_Code','string',''),
('Calc_Note','string',''),
#('Total_Cost','decimal(16,6)',''),
#('Sell_Price','decimal(20,6)',''),
('Crew_Size','string',''),
('Partsperhr','decimal(16,10)',''),
('Cycle_Time','decimal(10,4)',''),
('No_of_pcs','int',''),
('OEE','decimal(20,10)',''),
('Labor_Rate','decimal(20,10)',''),
('Variable_Burden_Rate','decimal(20,10)',''),
('Fixed_Burden_Rate','decimal(20,10)',''),
('Take_Rate','decimal(20,6)',''),
('Volume','int',''),
('Current_PO_Price','decimal(16,10)',''),
('ITC_Current_Cost','decimal(16,10)',''),
('zero_level_product_type','string',''),
('Customer_Part_No','string',''),
('lineage','string',''),
('lineage_sub','string',''),
('lineage_sub_desc','string',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),

    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "facton_factrevenue"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_GOLD",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 786bdf9f-4c26-46f7-8ea4-9957519b7df9, 9, Finished, Available, Finished, False)

# Process Table dimwrpart

In [8]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('wrpartKey',         'binary',  ''),
('wrpartKeyInt',         'bigint',  ''),
('PCN','int',''),
('Part_Key','int',''),
('Part_No','string',''),
('name','string',''),
('revision','string',''),
('Part_type','string',''),
('Part_status','string',''),
('unit','string',''),
('building_key','int',''),
('Part_group_key','string',''),
('Part_Group','string',''),
('Product_type_key','string',''),
('Product_type','string',''),
('Product_type_Code','string',''),
('Part_source_key','string',''),
('Purchased_Part','boolean',''),
('Manufactured_Part','boolean',''),
('Source','String',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),

    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "dimwrpart"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_GOLD",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 786bdf9f-4c26-46f7-8ea4-9957519b7df9, 10, Finished, Available, Finished, False)

## Process Table Bom_factrevenue

In [9]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('System','string',''),
('Version','string',''),
('Facton_Project_ID','int',''),
('PlantKey','binary',''),
('PlantKeyInt',         'bigint',  ''),
('bom_approved_workcenterKey',         'binary',  ''),
('bom_approved_workcenterKeyInt',         'bigint',  ''),
('exploded_bomKey',         'binary',  ''),
('exploded_bomKeyInt',         'bigint',  ''),
('part_masterKey',         'binary',  ''),
('part_masterKeyInt',         'bigint',  ''),
#('Location','string',''),
('Part_group_key','string',''),
('Product_Group','string',''),
('Product_type_key','string',''),
#('Customer','string',''),
('zero_level_desc','string',''),
('Zero_level_part_desc','string',''),
('level','string',''),
('parent_part_no','string',''),
('parent_part_no_desc','string',''),
('part_no','string',''),
('Description','string',''),
('Revision','string',''),
('Cost_Type','string',''),
('Cost_Sub_Type','string',''),
('Cost','decimal(16,6)',''),
('Quantity','decimal(16,6)',''),
('Operation_No','int',''),
('Operation_Code','string',''),
('Calc_Note','string',''),
#('Total_Cost','decimal(16,6)',''),
#('Sell_Price','decimal(20,6)',''),
('Crew_Size','string',''),
('Partsperhr','decimal(16,6)',''),
('Cycle_Time','decimal(10,4)',''),
('No_of_pcs','int',''),
('OEE','decimal(20,6)',''),
('Labor_Rate','decimal(20,6)',''),
('Variable_Burden_Rate','decimal(20,6)',''),
('Fixed_Burden_Rate','decimal(20,6)',''),
('Take_Rate','decimal(20,6)',''),
('Volume','int',''),
('Current_PO_Price','decimal(16,6)',''),
('ITC_Current_Cost','decimal(16,6)',''),
('zero_level_product_type','string',''),
('Customer_Part_No','string',''),
('lineage','string',''),
('lineage_sub','string',''),
('lineage_sub_desc','string',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),

    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "bom_factrevenue"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_GOLD",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 786bdf9f-4c26-46f7-8ea4-9957519b7df9, 11, Finished, Available, Finished, False)

## Process Table FactRevenue

In [6]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('System','string',''),
('Version','string',''),
('Facton_Project_ID','int',''),
('PlantKey','string',''),
('PlantKeyInt',         'bigint',  ''),
('approved_workcenterKey',         'binary',  ''),
('approved_workcenterKeyInt',         'bigint',  ''),
('exploded_bomKey',         'binary',  ''),
('exploded_bomKeyInt',         'bigint',  ''),
('part_masterKey',         'binary',  ''),
('part_masterKeyInt',         'bigint',  ''),
#('Location','string',''),
('Part_group_key','string',''),
('Product_Group','string',''),
('Product_type_key','string',''),
('Customer','string',''),
('zero_level_desc','string',''),
('level','string',''),
('parent_part_no','string',''),
('part_no','string',''),
('Description','string',''),
('Revision','string',''),
('Cost_Type','string',''),
('Cost_Sub_Type','string',''),
('Cost','decimal(16,6)',''),
('Quantity','decimal(16,6)',''),
('Operation_No','int',''),
('Operation_Code','string',''),
('Calc_Note','string',''),
#('Total_Cost','decimal(16,6)',''),
('Sell_Price','decimal(20,6)',''),
('Crew_Size','string',''),
('Partsperhr','decimal(16,6)',''),
('Cycle_Time','decimal(10,4)',''),
('No_of_pcs','int',''),
('OEE','decimal(20,10)',''),
('Labor_Rate','decimal(20,10)',''),
('Variable_Burden_Rate','decimal(20,10)',''),
('Fixed_Burden_Rate','decimal(20,10)',''),
('Take_Rate','decimal(20,6)',''),
('Volume','int',''),
('Current_PO_Price','decimal(16,6)',''),
('ITC_Current_Cost','decimal(16,6)',''),
('Customer_Part_No','string',''),
('Effective_Date','timestamp',''),
('Effective_DateKey',         'binary',  ''),
('Effective_DateKeyInt',         'bigint',  ''),
('expiration_date','timestamp',''),
('unit','string',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),

    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "factrevenue"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_GOLD",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 58e2e201-98ab-4ed2-bd8a-3fdddea3f493, 8, Finished, Available, Finished)

In [7]:
%%sql
DESCRIBE LH_BI_GOLD.factrevenue

StatementMeta(, 58e2e201-98ab-4ed2-bd8a-3fdddea3f493, 9, Finished, Available, Finished)

<Spark SQL result set with 50 rows and 3 fields>

## Process TestFact Revenue

In [8]:
# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('surrogate_key','String',''),
('System','string',''),
('Version','string',''),
('Facton_Project_ID','int',''),
('PlantKey','string',''),
('PlantKeyInt',         'bigint',  ''),
('approved_workcenterKey',         'binary',  ''),
('approved_workcenterKeyInt',         'bigint',  ''),
('exploded_bomKey',         'binary',  ''),
('exploded_bomKeyInt',         'bigint',  ''),
('part_masterKey',         'binary',  ''),
('part_masterKeyInt',         'bigint',  ''),
#('Location','string',''),
('Part_group_key','string',''),
('Product_Group','string',''),
('Product_type_key','string',''),
('Customer','string',''),
('zero_level_desc','string',''),
('level','string',''),
('parent_part_no','string',''),
('part_no','string',''),
('Description','string',''),
('Revision','string',''),
('Cost_Type','string',''),
('Cost_Sub_Type','string',''),
('Cost','decimal(16,6)',''),
('Quantity','decimal(16,6)',''),
('Operation_No','int',''),
('Operation_Code','string',''),
('Calc_Note','string',''),
#('Total_Cost','decimal(16,6)',''),
('Sell_Price','decimal(20,6)',''),
('Crew_Size','string',''),
('Partsperhr','decimal(16,6)',''),
('Cycle_Time','decimal(10,6)',''),
('No_of_pcs','int',''),
('OEE','decimal(10,6)',''),
('Labor_Rate','decimal(10,6)',''),
('Variable_Burden_Rate','decimal(10,6)',''),
('Fixed_Burden_Rate','decimal(10,6)',''),
('Take_Rate','decimal(10,6)',''),
('Volume','int',''),
('Current_PO_Price','decimal(16,6)',''),
('ITC_Current_Cost','decimal(16,6)',''),
('Customer_Part_No','string',''),
('Effective_Date','timestamp',''),
('Effective_DateKey',         'binary',  ''),
('Effective_DateKeyInt',         'bigint',  ''),
('expiration_date','timestamp',''),
('unit','string',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),

    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "testfactrevenue"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_GOLD",
    table_alias,
    comparisonArray,
    partitions,
    path
)

StatementMeta(, ac860065-0b5d-4634-b5de-ff9487144397, 10, Finished, Available, Finished)

## Process table DimPartMaster

In [6]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('part_masterKey',         'binary',  ''),
('part_masterKeyInt',         'bigint',  ''),
('PCN','int',''),
('Part_Key','int',''),
('Part_No','string',''),
('name','string',''),
('revision','string',''),
('Part_type','string',''),
('Part_status','string',''),
('unit','string',''),
('building_key','int',''),
('Part_group_key','int',''),
('Part_Group','string',''),
('Product_type_key','int',''),
('Product_type','string',''),
('Product_type_Code','string',''),
('Part_source_key','int',''),
('Purchased_Part','boolean',''),
('Manufactured_Part','boolean',''),
('Source','String',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "DimPartMaster"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_GOLD",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, facaf8bd-f658-4d3a-aaa2-bc55745ae5b8, 8, Finished, Available, Finished)

## 

#### Process table DimCommodity.

In [6]:
comparisonArray=[
	{"col_name":"CommodityKey", "data_type":"binary","comment":"[ID:{84947}]"}
	,{"col_name":"CommodityKeyInt", "data_type":"bigint","comment":"[ID:{84947INT}]"}
	,{"col_name":"PartNumber", "data_type":"string","comment":"[ID:{67469}]"}
	,{"col_name":"PlexPCN", "data_type":"string","comment":"[ID:{49948}]"}
	,{"col_name":"Commodity", "data_type":"string","comment":"[ID:{11613}]"}
	,{"col_name":"SubCommodity", "data_type":"string","comment":"[ID:{59576}]"}
	,{"col_name":"MainCommodityShort", "data_type":"string","comment":"[ID:{14413}]"}
	,{"col_name":"MainCommodityLong", "data_type":"string","comment":"[ID:{24554}]"}
	,{"col_name":"MaterialType", "data_type":"string","comment":"[ID:{46777}]"}
	,{"col_name":"QualityRelevant", "data_type":"string","comment":"[ID:{41278}]"}
	,{"col_name":"PartType", "data_type":"string","comment":"[ID:{11501}]"}
	,{"col_name":"MaterialContent", "data_type":"string","comment":"[ID:{74839}]"}
	,{"col_name":"SilverCommodityKey", "data_type":"binary","comment":"[ID:{28127}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"DimCommodity"

ProcessTable("LH_BI_GOLD","DimCommodity",comparisonArray,partitions,path)

StatementMeta(, 50115b92-8446-4c8a-b94f-270a37bd334f, 8, Finished, Available, Finished)

#### Process table DimFiscalDatePeriod.

In [7]:
comparisonArray=[
	{"col_name":"DateKey", "data_type":"binary","comment":"[ID:{93782}]"}
	,{"col_name":"DateKeyInt", "data_type":"bigint","comment":"[ID:{93782INT}]"}
	,{"col_name":"Date", "data_type":"date","comment":"[ID:{76525}]"}
	,{"col_name":"CalendarYear", "data_type":"smallint","comment":"[ID:{16658}]"}
	,{"col_name":"CalendarMonth", "data_type":"string","comment":"[ID:{28180}]"}
	,{"col_name":"MonthOfYear", "data_type":"smallint","comment":"[ID:{23698}]"}
	,{"col_name":"CalendarDay", "data_type":"string","comment":"[ID:{48396}]"}
	,{"col_name":"DayOfWeek", "data_type":"smallint","comment":"[ID:{93028}]"}
	,{"col_name":"IsWeekDay", "data_type":"boolean","comment":"[ID:{83949}]"}
	,{"col_name":"DayOfMonth", "data_type":"smallint","comment":"[ID:{47672}]"}
	,{"col_name":"IsLastDayofMonth", "data_type":"boolean","comment":"[ID:{73134}]"}
	,{"col_name":"DayOfYear", "data_type":"smallint","comment":"[ID:{74231}]"}
	,{"col_name":"WeekOfYearISO", "data_type":"smallint","comment":"[ID:{26820}]"}
	,{"col_name":"QuarterOfYear", "data_type":"smallint","comment":"[ID:{17213}]"}
	,{"col_name":"FiscalBeginDate", "data_type":"date","comment":"[ID:{17198}]"}
	,{"col_name":"FiscalEndDate", "data_type":"date","comment":"[ID:{78329}]"}
	,{"col_name":"FiscalMonthNumber", "data_type":"string","comment":"[ID:{77256}]"}
	,{"col_name":"FiscalMonth", "data_type":"string","comment":"[ID:{15957}]"}
	,{"col_name":"FiscalPeriod", "data_type":"smallint","comment":"[ID:{11616}]"}
	,{"col_name":"FiscalOrder", "data_type":"smallint","comment":"[ID:{70326}]"}
	,{"col_name":"QuarterGroup", "data_type":"string","comment":"[ID:{71840}]"}
	,{"col_name":"CalMonthBeginDate", "data_type":"date","comment":"[ID:{62532}]"}
	,{"col_name":"CalMonthEndDate", "data_type":"date","comment":"[ID:{44540}]"}
	,{"col_name":"SilverFiscalPeriodKey", "data_type":"binary","comment":"[ID:{45148}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"DimFiscalDatePeriod"

ProcessTable("LH_BI_GOLD","DimFiscalDatePeriod",comparisonArray,partitions,path)

StatementMeta(, 50115b92-8446-4c8a-b94f-270a37bd334f, 9, Finished, Available, Finished)

#### Process table DimPart.

In [8]:
comparisonArray=[
	{"col_name":"PartKey", "data_type":"binary","comment":"[ID:{92651}]"}
	,{"col_name":"PartKeyInt", "data_type":"bigint","comment":"[ID:{92651INT}]"}
	,{"col_name":"PartNumber", "data_type":"string","comment":"[ID:{65797}]"}
	,{"col_name":"PartSite", "data_type":"string","comment":"[ID:{32155}]"}
	,{"col_name":"PlexPartKey", "data_type":"int","comment":"[ID:{78545}]"}
	,{"col_name":"PlexPCN", "data_type":"string","comment":"[ID:{72335}]"}
	,{"col_name":"Unit", "data_type":"string","comment":"[ID:{42725}]"}
	,{"col_name":"Name", "data_type":"string","comment":"[ID:{16260}]"}
	,{"col_name":"SilverPartKey", "data_type":"binary","comment":"[ID:{53738}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"DimPart"

ProcessTable("LH_BI_GOLD","DimPart",comparisonArray,partitions,path)

StatementMeta(, 50115b92-8446-4c8a-b94f-270a37bd334f, 10, Finished, Available, Finished)

#### Process table DimPlant.

In [9]:
comparisonArray=[
	{"col_name":"PlantKey", "data_type":"binary","comment":"[ID:{45216}]"}
	,{"col_name":"PlantKeyInt", "data_type":"bigint","comment":"[ID:{45216INT}]"}
	,{"col_name":"PlantName", "data_type":"string","comment":"[ID:{11253}]"}
	,{"col_name":"ERPSite", "data_type":"string","comment":"[ID:{10713}]"}
	,{"col_name":"QADSite", "data_type":"string","comment":"[ID:{45304}]"}
	,{"col_name":"PCN", "data_type":"string","comment":"[ID:{60933}]"}
	,{"col_name":"PlexName", "data_type":"string","comment":"[ID:{93120}]"}
	,{"col_name":"ShortName", "data_type":"string","comment":"[ID:{67632}]"}
	,{"col_name":"Region", "data_type":"string","comment":"[ID:{68587}]"}
	,{"col_name":"ERP", "data_type":"string","comment":"[ID:{80029}]"}
	,{"col_name":"SilverPlantKey", "data_type":"binary","comment":"[ID:{79580}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"DimPlant"

ProcessTable("LH_BI_GOLD","DimPlant",comparisonArray,partitions,path)

StatementMeta(, 50115b92-8446-4c8a-b94f-270a37bd334f, 11, Finished, Available, Finished)

#### Process table DimSupplier.

In [10]:
comparisonArray=[
	{"col_name":"SupplierKey", "data_type":"binary","comment":"[ID:{61043}]"}
	,{"col_name":"SupplierKeyInt", "data_type":"bigint","comment":"[ID:{61043INT}]"}
	,{"col_name":"PlexPCN", "data_type":"string","comment":"[ID:{45494}]"}
	,{"col_name":"Site", "data_type":"string","comment":"[ID:{75694}]"}
	,{"col_name":"Currency", "data_type":"string","comment":"[ID:{59439}]"}
	,{"col_name":"PlexSupplierCode", "data_type":"string","comment":"[ID:{15826}]"}
	,{"col_name":"LegacySupplier", "data_type":"string","comment":"[ID:{76558}]"}
	,{"col_name":"SupplierName", "data_type":"string","comment":"[ID:{23944}]"}
	,{"col_name":"MultiPlantSupplierName", "data_type":"string","comment":"[ID:{56950}]"}
	,{"col_name":"SupplierType", "data_type":"string","comment":"[ID:{74746}]"}
	,{"col_name":"SilverSupplierKey", "data_type":"binary","comment":"[ID:{41196}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"DimSupplier"

ProcessTable("LH_BI_GOLD","DimSupplier",comparisonArray,partitions,path)

StatementMeta(, 50115b92-8446-4c8a-b94f-270a37bd334f, 12, Finished, Available, Finished)

#### Process table FactReceipt.

In [11]:
comparisonArray=[
	{"col_name":"ReceiptKey", "data_type":"binary","comment":"[ID:{62272}]"}
	,{"col_name":"ReceiptKeyInt", "data_type":"bigint","comment":"[ID:{62272INT}]"}
	,{"col_name":"PartKey", "data_type":"binary","comment":"[ID:{88499}]"}
	,{"col_name":"PartKeyInt", "data_type":"bigint","comment":"[ID:{88499INT}]"}
	,{"col_name":"PlantKey", "data_type":"binary","comment":"[ID:{21903}]"}
	,{"col_name":"PlantKeyInt", "data_type":"bigint","comment":"[ID:{21903INT}]"}
	,{"col_name":"CommodityKey", "data_type":"binary","comment":"[ID:{58150}]"}
	,{"col_name":"CommodityKeyInt", "data_type":"bigint","comment":"[ID:{58150INT}]"}
	,{"col_name":"SupplierKey", "data_type":"binary","comment":"[ID:{19592}]"}
	,{"col_name":"SupplierKeyInt", "data_type":"bigint","comment":"[ID:{19592INT}]"}
	,{"col_name":"ReceivedQuantity", "data_type":"double","comment":"[ID:{16458}]"}
	,{"col_name":"ReceivedDate", "data_type":"date","comment":"[ID:{97018}]"}
	,{"col_name":"ReceivedUM", "data_type":"string","comment":"[ID:{97019}]"}
	,{"col_name":"SilverReceiptKey", "data_type":"binary","comment":"[ID:{63017}]"}
	,{"col_name":"FiscalDatePeriodKey", "data_type":"binary","comment":"[ID:{19234}]"}
	,{"col_name":"FiscalDatePeriodKeyInt", "data_type":"bigint","comment":"[ID:{19234INT}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"FactReceipt"

ProcessTable("LH_BI_GOLD","FactReceipt",comparisonArray,partitions,path)

StatementMeta(, 50115b92-8446-4c8a-b94f-270a37bd334f, 13, Finished, Available, Finished)

#### Process table CustomerAddress

In [12]:
comparisonArray=[
	{"col_name":"customeraddressKey", "data_type":"binary","comment":"[ID:{84947}]"}
	,{"col_name":"customeraddressKeyInt", "data_type":"bigint","comment":"[ID:{84947INT}]"}
	,{"col_name":"PCN", "data_type":"string","comment":"[ID:{67469}]"}
	,{"col_name":"customer_no", "data_type":"string","comment":"[ID:{49948}]"}
	,{"col_name":"customer_code", "data_type":"string","comment":"[ID:{11613}]"}
	,{"col_name":"Name", "data_type":"string","comment":"[ID:{59576}]"}
	,{"col_name":"customer_Status", "data_type":"string","comment":"[ID:{14413}]"}
	,{"col_name":"customer_type", "data_type":"string","comment":"[ID:{24554}]"}
	,{"col_name":"customer_Address_no", "data_type":"string","comment":"[ID:{46777}]"}
	,{"col_name":"Customer_Address_code", "data_type":"string","comment":"[ID:{41278}]"}
	,{"col_name":"SilvercustomeraddressKey", "data_type":"binary","comment":"[ID:{28127}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"Dimcustomeraddress"

ProcessTable("LH_BI_GOLD","Dimcustomeraddress",comparisonArray,partitions,path)

StatementMeta(, 50115b92-8446-4c8a-b94f-270a37bd334f, 14, Finished, Available, Finished)

#### Process table factsales

In [6]:
comparisonArray=[
    {"col_name":"SalesKey", "data_type":"binary","comment":"[ID:{98318}]"}
	,{"col_name":"SalesKeyInt", "data_type":"bigint","comment":"[ID:{89366}]"}
	,{"col_name":"PartKey", "data_type":"binary","comment":"[ID:{35258}]"}
	,{"col_name":"PartKeyInt", "data_type":"bigint","comment":"[ID:{42803}]"}
	,{"col_name":"PlantKey", "data_type":"binary","comment":"[ID:{74139}]"}
	,{"col_name":"PlantKeyInt", "data_type":"bigint","comment":"[ID:{61521}]"}
	,{"col_name":"CurrencyKey", "data_type":"binary","comment":"[ID:{76698}]"}
	,{"col_name":"CurrencyKeyInt", "data_type":"bigint","comment":"[ID:{48771}]"}
	,{"col_name":"CustomerAddressKey", "data_type":"binary","comment":"[ID:{26857}]"}
	,{"col_name":"CustomerAddressKeyInt", "data_type":"bigint","comment":"[ID:{28979}]"}
	,{"col_name":"Invoice_No", "data_type":"string","comment":"[ID:{66242}]"}
	,{"col_name":"Account", "data_type":"string","comment":"[ID:{76568}]"}
	,{"col_name":"BillTo", "data_type":"string","comment":"[ID:{76569}]"}
	,{"col_name":"ShipTo", "data_type":"string","comment":"[ID:{76570}]"}
	,{"col_name":"Ship_date", "data_type":"date","comment":"[ID:{76571}]"}
	,{"col_name":"shipper_no", "data_type":"string","comment":"[ID:{76572}]"}
	,{"col_name":"Supplier", "data_type":"string","comment":"[ID:{76573}]"}
	,{"col_name":"customer_po", "data_type":"string","comment":"[ID:{76574}]"}
	,{"col_name":"PO_No", "data_type":"string","comment":"[ID:{76575}]"}
	,{"col_name":"Customer_part", "data_type":"string","comment":"[ID:{76576}]"}
	,{"col_name":"Invoice_Date", "data_type":"date","comment":"[ID:{76577}]"}
	,{"col_name":"Invoice_Line", "data_type":"string","comment":"[ID:{76578}]"}
	,{"col_name":"Quantity", "data_type":"string","comment":"[ID:{76579}]"}
	,{"col_name":"Price", "data_type":"string","comment":"[ID:{76580}]"}
	,{"col_name":"Invoice_amount", "data_type":"string","comment":"[ID:{76581}]"}
	,{"col_name":"Currency", "data_type":"string","comment":"[ID:{76582}]"}
	,{"col_name":"SilverSalesKey", "data_type":"binary","comment":"[ID:{76583}]"}
	,{"col_name":"FiscalDatePeriodKey", "data_type":"binary","comment":"[ID:{76584}]"}
	,{"col_name":"FiscalDatePeriodKeyInt", "data_type":"bigint","comment":"[ID:{76585}]"}	
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}

]
partitions=[]
path=destination_filepath+"FactSales"

ProcessTable("LH_BI_GOLD","FactSales",comparisonArray,partitions,path)


StatementMeta(, f5ed965b-172d-4cad-ad61-21490aee253b, 8, Finished, Available, Finished)

#### Process table Currency

In [14]:
comparisonArray=[
    {"col_name":"currencykey", "data_type":"binary","comment":"[ID:{72462}]"}
    ,{"col_name":"currencykeyint", "data_type":"bigint","comment":"[ID:{58399}]"}
    ,{"col_name":"silvercurrencykey", "data_type":"binary","comment":"[ID:{30272}]"}
    ,{"col_name":"plexcurrencykey", "data_type":"int","comment":"[ID:{95930}]"}
    ,{"col_name":"currencysymbol", "data_type":"string","comment":"[ID:{12220}]"}
    ,{"col_name":"currencyhtml", "data_type":"string","comment":"[ID:{66355}]"}
    ,{"col_name":"exchangerate", "data_type":"double","comment":"[ID:{17808}]"}
    ,{"col_name":"exchangeratedate", "data_type":"date","comment":"[ID:{55358}]"}
    ,{"col_name":"currencycode", "data_type":"string","comment":"[ID:{1234}]"}
    ,{"col_name":"description", "data_type":"string","comment":"[ID:{42675}]"}
    ,{"col_name":"sortorder", "data_type":"int","comment":"[ID:{48310}]"}
    ,{"col_name":"updatedate", "data_type":"date","comment":"[ID:{99791}]"}
    ,{"col_name":"active", "data_type":"boolean","comment":"[ID:{55598}]"}
    ,{"col_name":"currencyunicode", "data_type":"string","comment":"[ID:{97720}]"}
    ,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
    ]

partitions=[]
path=destination_filepath+"DimCurrency"

ProcessTable("LH_BI_GOLD","DimCurrency",comparisonArray,partitions,path)    


StatementMeta(, 50115b92-8446-4c8a-b94f-270a37bd334f, 16, Finished, Available, Finished)

#### Process table Audit_RuntimeTableLog.

In [15]:
comparisonArray=[
	{"col_name":"GroupingID", "data_type":"string","comment":""}
    ,{"col_name":"StartDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"DatabaseName", "data_type":"string","comment":""}
    ,{"col_name":"TableName", "data_type":"string","comment":""}
    ,{"col_name":"TableStartTime", "data_type":"timestamp","comment":""}
    ,{"col_name":"TableEndTime", "data_type":"timestamp","comment":""}
    ,{"col_name":"TableDurationSeconds", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsAffected", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsInserted", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsUpdated", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsDeleted", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsBefore", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsAfter", "data_type":"bigint","comment":""}
    ,{"col_name":"Audit_Status", "data_type":"string","comment":""}
    ,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=["StartDateTime","DatabaseName","TableName"]

path=destination_filepath+"Audit_RuntimeTableLog"

ProcessTable("LH_BI_GOLD","Audit_RuntimeTableLog",comparisonArray,partitions,path)

StatementMeta(, 50115b92-8446-4c8a-b94f-270a37bd334f, 17, Finished, Available, Finished)

#### End of Notebook